# Vectorized Miller Plane And Direction Workflows

This tutorial shows the intended PyTex workflow for first-class Miller plane and direction objects attached to a phase.

It covers:

- vectorized creation from arrays
- hexagonal index conversion
- cartesian vectors, normals, and d-spacings
- angle matrices
- symmetry-family expansion
- projecting directions onto planes


In [ ]:
from __future__ import annotations

import numpy as np

from pytex import (
    FrameDomain,
    Handedness,
    Lattice,
    MillerDirectionSet,
    MillerPlaneSet,
    Phase,
    ReferenceFrame,
    SymmetrySpec,
    angle_dir_dir_rad,
    project_directions_onto_planes,
)


In [ ]:
crystal = ReferenceFrame(
    name="crystal",
    domain=FrameDomain.CRYSTAL,
    axes=("a", "b", "c"),
    handedness=Handedness.RIGHT,
)
lattice = Lattice(3.6, 3.6, 3.6, 90.0, 90.0, 90.0, crystal_frame=crystal)
symmetry = SymmetrySpec.from_point_group("m-3m", reference_frame=crystal)
phase = Phase("fcc_demo", lattice=lattice, symmetry=symmetry, crystal_frame=crystal)
phase


In [ ]:
planes = MillerPlaneSet.from_hkl(
    np.array([[1, 0, 0], [1, 1, 0], [1, 1, 1]], dtype=np.int64),
    phase=phase,
)
directions = MillerDirectionSet.from_uvw(
    np.array([[1, 0, 0], [1, 1, 0], [1, 1, 1]], dtype=np.int64),
    phase=phase,
)

print("HKL -> HKIL:\n", planes.to_hkil())
print("UVW -> UVTW:\n", directions.to_UVTW())


In [ ]:
table = np.column_stack(
    [
        planes.indices,
        np.round(planes.normals_cartesian(), 6),
        np.round(planes.d_spacings_angstrom(), 6)[:, None],
    ]
)
print("Columns: h k l | nx ny nz | d(angstrom)")
print(table)


In [ ]:
angle_matrix = directions.angle_matrix_rad(antipodal=True)
reference = MillerDirectionSet.from_uvw([[1, 0, 0]], phase=phase)
pairwise = angle_dir_dir_rad(directions, reference, antipodal=True)
print("Direction angle matrix (rad):\n", np.round(angle_matrix, 6))
print("Angles to <100> (rad):\n", np.round(pairwise, 6))


In [ ]:
equivalent_indices, mask = planes.symmetry_equivalent_indices()
first_family = equivalent_indices[0, mask[0]]
print("First plane family under cubic symmetry:\n", first_family)


In [ ]:
projection_planes = MillerPlaneSet.from_hkl([[0, 0, 1], [1, 1, 1], [0, 1, 0]], phase=phase)
projected, degenerate = project_directions_onto_planes(directions, projection_planes)
print("Projected direction vectors:\n", np.round(projected, 6))
print("Degenerate mask:\n", degenerate)


The vectorized `symmetry_equivalent_indices()` surface is designed for large batches and benchmarking.
Use `symmetry_equivalents()` when you want a tuple of semantic sets for interactive work.
